In [1]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
import pyarrow
import fastparquet
import pandas as pd

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 07:47:21] INFO     Using                                                                  ]8;id=357890;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py\__init__.py]8;;\:]8;id=699412;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\framework\project\__init__.py#269\269]8;;\
                             'c:\anaconda3\envs\central\Lib\site-packages\kedro\framework\project\r                
                             ich_logging.yml' as logging configuration.                                            

[04/21/26 07:47:23] WARNING  c:\anaconda3\envs\central\Lib\site-packages\requests\__init__.py:113:  ]8;id=288207;file://c:\anaconda3\envs\central\Lib\warnings.py\warnings.py]8;;\:]8;id=294062;file://c:\anaconda3\envs\central\Lib\warnings.py#110\110]8;;\
                             RequestsDependencyWarning: urllib3 (2.6.3) or chardet                                 
                             (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported                    
                             version!                                                                              
                               warnings.warn(                                                                      
                                                                                                                   

[04/21/26 07:47:24] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=29264;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=72835;file://c:\anaconda3\envs\central\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\CENTRAL\central-perm-flow


# Importar nodos

In [2]:
# 2. Importar todas las funciones del archivo nodes.py
import central_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))


['Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'central_preprocessing_calaca', 'central_preprocessing_estaca', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata']


In [20]:
# Muestra la ayuda formateada
help(nodes.select_columns)

# O simplemente imprime el texto de la descripción
print(nodes.select_columns.__doc__)

Help on function select_columns in module central_perm_flow.pipelines.data_processing.nodes:

select_columns(df: pandas.DataFrame, columns: list) -> pandas.DataFrame
    Selecciona un subconjunto de columnas del DataFrame.
    Args:
        df: El DataFrame del cual seleccionar columnas.
        columns: Una lista de nombres de columnas a seleccionar.
    Returns:
        pd.DataFrame: Un nuevo DataFrame que contiene solo las columnas seleccionadas.

Selecciona un subconjunto de columnas del DataFrame.
Args:
    df: El DataFrame del cual seleccionar columnas.
    columns: Una lista de nombres de columnas a seleccionar.
Returns:
    pd.DataFrame: Un nuevo DataFrame que contiene solo las columnas seleccionadas.




# Fuentes de Información

## Base de Estados de los Estudiantes

In [ ]:
# Base de central con fechas de bajas, fechas de graduación
central_caracterizacion = catalog.load('central_caracterizacion')

### Parámetros

In [ ]:
central_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
central_col_emails = ['correo','correo_electronico_del_contacto_emergencia']
central_col_dd = ['identidad']
central_col_sort = ['identidad']

In [ ]:
# Limpieza de la base de operaciones
central_caracterizacion = nodes.clean_column_names(central_caracterizacion)
central_caracterizacion = nodes.convert_all_standardized_dates(central_caracterizacion,central_col_fechas)
central_caracterizacion['identidad'] = pd.to_numeric(central_caracterizacion['identidad'], errors='coerce')
central_caracterizacion = nodes.clean_column_objects(central_caracterizacion,central_col_emails)
central_caracterizacion_sd, central_caracterizacion_cd  = nodes.check_and_export_duplicates(central_caracterizacion, subset=central_col_dd, col_ordenar = central_col_sort)
central_caracterizacion = central_caracterizacion.drop(columns='nivel') 



## Calendario Académico

In [ ]:
# Calendario Académico
central_estados_calac = catalog.load("central_estados_calac")

### Parámetros

In [ ]:
central_caracterizacion = central_estados_calac.merge(central_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])

In [ ]:

# Base de central con fechas de bajas, fechas de graduación
central_caracterizacion = catalog.load('central_caracterizacion')
central_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
central_col_emails = ['correo','correo_electronico_del_contacto_emergencia']
central_col_dd = ['identidad']
central_col_sort = ['identidad']
# Limpieza de la base de operaciones
central_caracterizacion = nodes.clean_column_names(central_caracterizacion)
central_caracterizacion = nodes.convert_all_standardized_dates(central_caracterizacion,central_col_fechas)
central_caracterizacion['identidad'] = pd.to_numeric(central_caracterizacion['identidad'], errors='coerce')
central_caracterizacion = nodes.clean_column_objects(central_caracterizacion,central_col_emails)
central_caracterizacion_sd, central_caracterizacion_cd  = nodes.check_and_export_duplicates(central_caracterizacion, subset=central_col_dd, col_ordenar = central_col_sort)
central_caracterizacion = central_caracterizacion.drop(columns='nivel') 

# Calendario Académico
central_estados_calac = catalog.load("central_estados_calac")
central_caracterizacion = central_estados_calac.merge(central_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])





In [ ]:
import pandas as pd

def transformar_caracterizacion_central(
    df_caracterizacion: pd.DataFrame,
    df_estados_calac: pd.DataFrame,
    params_fechas: list,
    params_emails: list,
    params_duplicados: list,
    params_orden: list,
    params_columnas_caracterizacion: list
) -> pd.DataFrame:
    """
    Función para limpiar la base de operaciones y cruzarla con el calendario académico.
    """
    
    # 1. Limpieza de nombres de columnas
    df = nodes.clean_column_names(df_caracterizacion)
    
    # 2. Conversión de fechas usando los parámetros del YML
    df = nodes.convert_all_standardized_dates(df, params_fechas)
    
    # 3. Conversión de identidad a numérico (Coerce para manejar errores)
    df['identidad'] = pd.to_numeric(df['identidad'], errors='coerce')
    
    # 4. Limpieza de correos/objetos
    df = nodes.clean_column_objects(df, params_emails)
    
    # 5. Manejo de duplicados (obtenemos la base sin duplicados 'sd')
    df_sd, _ = nodes.check_and_export_duplicates(
        df, 
        subset=params_duplicados, 
        col_ordenar=params_orden
    )
    
    # 6. Eliminar columna 'nivel' si existe
    df_sd = df_sd.drop(columns='nivel', errors='ignore')

    # 7. Seleccionar columnas especificas
    df_sd = nodes.select_columns(df_sd, params_columnas_caracterizacion)

    
    # 8. Cruce con Calendario Académico (Merge)
    # Nota: Usamos df_sd para asegurar que el cruce no duplique filas inesperadamente
    resultado = df_estados_calac.merge(
        df_sd, 
        how='left', 
        left_on=['identificacion'], 
        right_on=['identidad']
    )
    
    return resultado

In [5]:
parameters = catalog.load("parameters")
parameters['central_caracterizacion_params']

[04/21/26 07:47:47] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=495622;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=836619;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\


{
    'col_fechas': ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases'],
    'col_emails': ['correo', 'correo_electronico_del_contacto_emergencia'],
    'col_dd': ['identidad'],
    'col_sort': ['identidad']
}

In [7]:
# Kedro carga automáticamente la variable 'catalog'
df_caracterizacion = catalog.load("central_caracterizacion")
df_estados_calac = catalog.load("central_estados_calac")


[04/21/26 07:48:04] INFO     Loading data from central_caracterizacion (CSVDataset)...         ]8;id=220386;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=363086;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

[04/21/26 07:48:05] INFO     Loading data from central_estados_calac (ExcelDataset)...         ]8;id=989225;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=485590;file://c:\anaconda3\envs\central\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [17]:
df_estados_calac_caracterizacion = transformar_caracterizacion_central(
    df_caracterizacion=df_caracterizacion,
    df_estados_calac=df_estados_calac,
    params_fechas=parameters['central_caracterizacion_params']['col_fechas'],
    params_emails=parameters['central_caracterizacion_params']['col_emails'],
    params_duplicados=parameters['central_caracterizacion_params']['col_dd'],
    params_orden=parameters['central_caracterizacion_params']['col_sort']
)

In [ ]:
from kedro.pipeline import Pipeline, node, pipeline
from .nodes import transformar_caracterizacion_central

def create_pipeline(**kwargs) -> Pipeline:
    return pipeline(
        [
            node(
                func=transformar_caracterizacion_central,
                inputs=[
                    "central_caracterizacion",      # Dataset del catalog.yml
                    "central_estados_calac",       # Dataset del catalog.yml
                    "params:central_processing.col_fechas",
                    "params:central_processing.col_emails",
                    "params:central_processing.col_dd",
                    "params:central_processing.col_sort",
                ],
                outputs="central_caracterizacion_final", # Nombre del resultado
                name="nodo_transformar_central",
            ),
        ]
    )

## EDA

In [ ]:
df_estados_calac_caracterizacion.columns


Index(['identificacion', 'codigo_sis', 'id_inconcert', 'nombres',
       'usuario_institucional', 'alianza', 'cohorte', 'cohorte_inicial',
       'cohorte_actual', 'nivel',
       ...
       'complemento', 'titulo_profesional_pregrado', 'ano_de_graduacion',
       'universidad_que_titula', '_ha_cursado_posgrados_adicionales_',
       'titulo_obtenido', 'fecha_graduacion',
       'universidad_que_otorga_el_titulo',
       '_ha_cursado_posgrados_en_la_universidad_central_', '_cual_'],
      dtype='str', length=123)

In [18]:
df_estados_calac_caracterizacion.loc[:,['genero','estado']].value_counts()


genero     estado              
femenino   activo                  742
masculino  activo                  680
femenino   egresado no graduado    105
           baja definitiva          77
masculino  baja definitiva          61
           egresado no graduado     40
           baja temporal            24
femenino   baja temporal            24
Name: count, dtype: int64